# ALPR — Colab bootstrap

Clones the repo into a fresh T4 runtime, installs the package, and verifies the GPU.

Run this first in every new Colab session. Every other notebook assumes it has run.

**Before running:** `Runtime > Change runtime type > T4 GPU`.

## 1. Check the runtime *before* installing anything

A CPU runtime installs the CUDA wheels perfectly happily and then fails hours later, mid-training. Fail here instead.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import sys

print("python", sys.version.split()[0])

## 2. Clone and install

Set `BRANCH` to the phase branch you are working on. The repo is public, so no token is needed to read it.

In [ ]:
REPO = "https://github.com/fayazhussain2821/Automatic-License-Plate-Recognition.git"
BRANCH = "main"

import os

if os.path.isdir("/content/ALPR"):
    !cd /content/ALPR && git fetch --quiet origin && git checkout --quiet $BRANCH && git pull --quiet
else:
    !git clone --quiet --branch $BRANCH $REPO /content/ALPR

%cd /content/ALPR
!git log --oneline -1

In [ ]:
# Editable install so `git pull` picks up code changes without reinstalling.
#
# Base install only. The `gpu` extra (PaddleOCR) is deliberately NOT
# installed here: nothing before Phase 4 uses OCR, and pulling it in early
# breaks the install. PyPI's paddlepaddle-gpu is frozen at 2.6.2 while
# current paddleocr needs PaddlePaddle 3.x, whose GPU wheels live on
# Paddle's own index rather than PyPI, so pip cannot resolve the pair.
# Phase 4 pins the working combination against the right index.
!pip install --quiet -e .

# An editable install registers itself through a .pth file in site-packages,
# and .pth files are only processed at interpreter startup. This kernel was
# already running when pip ran, so it cannot see the package — `import alpr`
# fails with ModuleNotFoundError even though the install above succeeded.
# Adding the source directory explicitly avoids needing a runtime restart.
import sys

SRC = "/content/ALPR/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import alpr

print(f"alpr {alpr.__version__} importable")

## 3. Verify

`require_gpu()` raises if the runtime has no CUDA device — this is the assertion every training notebook opens with.

In [ ]:
from alpr.env import in_colab, require_gpu

gpu = require_gpu()
print(f"colab: {in_colab()}")
print(f"gpu:   {gpu}")

assert "T4" in gpu.name, f"expected a T4, got {gpu.name!r} — check Runtime > Change runtime type"
print("\nready.")

In [ ]:
!alpr env